# Step-Free London — Notebook 02: which lines trap their users?

Stations don't exist in isolation — you travel along a *line*. A wheelchair
user on a line where half the stations have stairs can only make partial
journeys. Here we break accessibility down line by line.

In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../data/processed/tube_stepfree_final.csv")

lines = df.copy()
lines["line"] = lines["lines"].str.split(", ")
lines = lines.explode("line")
lines = lines[lines["line"].notna()]

tube_lines = {
    "Bakerloo", "Central", "Circle", "District", "Elizabeth line",
    "Hammersmith & City", "Jubilee", "Metropolitan", "Northern",
    "Piccadilly", "Victoria", "Waterloo & City",
}
dropped = sorted(set(lines["line"]) - tube_lines)
print(f"non-tube line entries dropped ({len(dropped)}): {dropped[:8]}...")
lines = lines[lines["line"].isin(tube_lines)]
print(f"station-line pairs: {len(lines)} across {lines['line'].nunique()} tube lines")


non-tube line entries dropped (15): ['137', '167', '19', '20', '22', '397', '452', '462']...
station-line pairs: 384 across 11 tube lines


In [2]:
status_by_line = (
    lines.groupby("line")["step_free"]
    .value_counts(normalize=True)
    .unstack()
    .fillna(0)
    * 100
).round(1)
for col in ["step-free", "partial", "not step-free"]:
    if col not in status_by_line.columns:
        status_by_line[col] = 0.0
status_by_line["stations"] = lines.groupby("line").size()
status_by_line.sort_values("step-free", ascending=False)


step_free,not step-free,partial,step-free,stations
line,,,,
Jubilee,29.6,7.4,63.0,27
Waterloo & City,0.0,50.0,50.0,2
Victoria,25.0,31.2,43.8,16
District,48.3,11.7,40.0,60
Hammersmith & City,44.8,17.2,37.9,29
Piccadilly,58.5,5.7,35.8,53
Northern,61.5,7.7,30.8,52
Circle,58.3,11.1,30.6,36
Metropolitan,40.0,31.4,28.6,35


In [3]:
plot_df = status_by_line.reset_index().melt(
    id_vars=["line", "stations"],
    value_vars=["step-free", "partial", "not step-free"],
    var_name="status",
    value_name="pct",
)
order = status_by_line.sort_values("not step-free", ascending=False).index.tolist()
fig = px.bar(
    plot_df,
    y="line",
    x="pct",
    color="status",
    category_orders={"line": order, "status": ["step-free", "partial", "not step-free"]},
    color_discrete_map={
        "step-free": "#2e7d32",
        "partial": "#f9a825",
        "not step-free": "#c62828",
    },
    orientation="h",
    title="Tube line accessibility (% of stations)",
    labels={"pct": "% of stations", "line": ""},
    height=560,
)
fig.update_layout(barmode="stack", legend_title="")
fig.show()


**Reading the chart:** deep-level tube lines bored in the Victorian era sit at
the bottom — their narrow tunnels sit too deep for cheap lift installs.
Sub-surface lines (Circle, District, Metropolitan — cut-and-dug near the
surface) and modern builds dominate the top. The infrastructure of 1863 still
decides who can travel in 2026.